In [ ]:
!hf download celesca/urine-zip --repo-type dataset --local-dir ./

Fetching 5 files:   0% 0/5 [00:00<?, ?it/s]Downloading 'new_images.zip' to '.cache/huggingface/download/RMT4hqhRPv-W1Noxg0qCmu7q39c=.2248ce5b4ea7e877111e66fc7c83317053290ff8cbad1ab3cec2ee156227c2ac.incomplete'

new_images.zip:   0% 0.00/7.83G [00:00<?, ?B/s]

urine_data.xlsx:   0% 0.00/1.75M [00:00<?, ?B/s]


processed_images.zip:   0% 0.00/15.0M [00:00<?, ?B/s]



README.md: 100% 31.0/31.0 [00:00<00:00, 181kB/s]
Download complete. Moving file to README.md




.gitattributes: 2.51kB [00:00, 4.34MB/s]
Download complete. Moving file to .gitattributes
Fetching 5 files:  20% 1/5 [00:00<00:03,  1.33it/s]

urine_data.xlsx: 100% 1.75M/1.75M [00:03<00:00, 567kB/s]
Download complete. Moving file to urine_data.xlsx

new_images.zip:   0% 8.13k/7.83G [00:03<1012:15:31, 2.15kB/s]
new_images.zip:   0% 22.8k/7.83G [00:04<306:02:03, 7.10kB/s] 
new_images.zip:   0% 48.0k/7.83G [00:04<120:34:44, 18.0kB/s]
new_images.zip:   0% 184k/7.83G [00:04<21:54:37, 99.2kB/s]  
new_images.zip:   0% 246k/7.83G [00:04

In [ ]:
!unzip new_images.zip

Archive:  new_images.zip
   creating: AI/
   creating: AI/065/
  inflating: AI/065/065_Honor X9C_F3.jpg  
  inflating: AI/065/065_Honor X9C_L3.jpg  
  inflating: AI/065/065_Honor X9C_R3.jpg  
  inflating: AI/065/065_Honor X9C_U3.jpg  
  inflating: AI/065/065_iPhone 13_F1.jpg  
  inflating: AI/065/065_iPhone 13_L1.jpg  
  inflating: AI/065/065_iPhone 13_R1.jpg  
  inflating: AI/065/065_iPhone 13_U1.jpg  
  inflating: AI/065/065_Realme 7 5G_F4.jpg  
  inflating: AI/065/065_Realme 7 5G_L4.jpg  
  inflating: AI/065/065_Realme 7 5G_R4.jpg  
  inflating: AI/065/065_Realme 7 5G_U4.jpg  
  inflating: AI/065/065_Samsung Galaxy S22 Ultra_F2.jpg  
  inflating: AI/065/065_Samsung Galaxy S22 Ultra_L2.jpg  
  inflating: AI/065/065_Samsung Galaxy S22 Ultra_R2.jpg  
  inflating: AI/065/065_Samsung Galaxy S22 Ultra_U2.jpg  
   creating: AI/066/
  inflating: AI/066/066_Honor X9C_F3.jpg  
  inflating: AI/066/066_Honor X9C_L3.jpg  
  inflating: AI/066/066_Honor X9C_R3.jpg  
  inflating: AI/066/066_Honor X

In [ ]:
import pandas as pd
import numpy as np
import os

df = pd.read_excel("urine_data.xlsx", sheet_name="AI Training")

# Filter the DataFrame to exclude rows where 'Name' contains "iPhone"
df = df[~df['Name'].str.contains("iPhone")]

In [ ]:
df

,Name,Position Mark,R,G,B,Sp.Refractometer
1,065_Samsung Galaxy S22 Ultra_F2.jpg,F2,214,191,113,1.025
2,065_Honor X9C_F3.jpg,F3,185,162,92,1.025
3,065_Realme 7 5G_F4.jpg,F4,173,147,62,1.025
5,065_Samsung Galaxy S22 Ultra_L2.jpg,L2,170,145,68,1.025
6,065_Honor X9C_L3.jpg,L3,164,142,69,1.025
...,...,...,...,...,...,...
4250,370_Honor X9C_R3.jpg,R3,157,149,136,1.002
4251,370_Realme 7 5G_R4.jpg,R4,141,142,127,1.002
4253,370_Samsung Galaxy S22 Ultra_U2.jpg,U2,182,181,165,1.002
4254,370_Honor X9C_U3.jpg,U3,157,150,134,1.002


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

class UrineDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_filename = self.dataframe.iloc[idx, 0]
        subject_id = img_filename.split('_')[0] # Extract subject ID from filename
        img_name = os.path.join(self.img_dir, subject_id, img_filename) # Construct the correct image path
        image = Image.open(img_name).convert('RGB')
        label = self.dataframe.iloc[idx, 5] # Assuming 'Sp.Refractometer' is the 6th column (index 5)

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)

# Define transformations for training with augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)), # Resize images to a fixed size
    transforms.RandomRotation(10), # Randomly rotate by up to 10 degrees
    transforms.RandomHorizontalFlip(), # Randomly flip the image horizontally
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), # Randomly change brightness, contrast, saturation, and hue
    transforms.ToTensor(), # Convert images to PyTorch tensors
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalize images
])

# Define transformations for testing (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)), # Resize images to a fixed size
    transforms.ToTensor(), # Convert images to PyTorch tensors
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalize images
])


# Create the dataset and dataloader
# Assuming the images are in a directory named 'AI'
img_directory = 'AI'
# Create separate datasets for training and testing with different transforms
train_dataset_full = UrineDataset(df, img_directory, transform=train_transform)
test_dataset_full = UrineDataset(df, img_directory, transform=test_transform)

# Note: We will split the full datasets into train/test sets in the next cell

print("Dataset created with {} samples.".format(len(train_dataset_full)))

Dataset created with 3192 samples.


In [ ]:
from torch.utils.data import random_split

# Define the split ratio (e.g., 80% for training, 20% for testing)
train_ratio = 0.9
test_ratio = 1 - train_ratio
train_size = int(train_ratio * len(train_dataset_full)) # Use the dataset with training transforms
test_size = len(test_dataset_full) - train_size # Use the dataset with testing transforms

# Split the dataset
train_dataset, test_dataset = random_split(train_dataset_full, [train_size, test_size]) # Split the training dataset with augmentation
# Note: We are splitting the full dataset with training transforms for the training set, and the full dataset with testing transforms for the test set.

# Create DataLoaders for training and testing sets
# Increase num_workers for potentially faster data loading
num_workers = 2 # You can adjust this number based on your system's capabilities
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=num_workers)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=num_workers)

print("Training dataset size: {}".format(len(train_dataset)))
print("Testing dataset size: {}".format(len(test_dataset)))

Training dataset size: 2872
Testing dataset size: 320


In [ ]:
import torch
import torch.nn as nn
import timm

# Define the CNN model using ConvNextV2 from timm
class CNNRegressor(nn.Module):
    def __init__(self, model_name='timm/convnextv2_base.fcmae_ft_in22k_in1k', pretrained=True):
        super(CNNRegressor, self).__init__()
        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=0) # Load model without the final classification layer
        self.regressor = nn.Linear(self.model.num_features, 1) # Add a linear layer for regression

    def forward(self, x):
        x = self.model(x)
        x = self.regressor(x)
        return x

# Instantiate the model
model = CNNRegressor()

# Define loss function and optimizer
criterion = nn.MSELoss() # Mean Squared Error for regression
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # Adam optimizer

print("CNN Regressor model created.")

CNN Regressor model created.


In [ ]:
# Check if GPU is available and move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

num_epochs = 2

print(f"Training the model on {device} for {num_epochs} epochs...")

from tqdm.auto import tqdm

for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    # Wrap train_dataloader with tqdm for progress tracking
    for images, labels in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels = images.to(device), labels.to(device).unsqueeze(1) # Move data to device and add a dimension to labels

        # Scale the labels
        scaled_labels = (labels * 1000) - 1000

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, scaled_labels)  # Use scaled labels for loss calculation

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

print("Training finished.")

# Evaluate the model
model.eval()  # Set the model to evaluation mode
test_loss = 0.0
with torch.no_grad(): # Disable gradient calculation during evaluation
    for images, labels in test_dataloader:
        images, labels = images.to(device), labels.to(device).unsqueeze(1) # Move data to device and add a dimension to labels

        # Scale the labels for evaluation
        scaled_labels = (labels * 1000) - 1000

        outputs = model(images)
        loss = criterion(outputs, scaled_labels) # Use scaled labels for loss calculation
        test_loss += loss.item() * images.size(0)

test_loss /= len(test_dataset)

print(f"Test Loss (MSE): {test_loss:.4f}")

Training the model on cuda for 2 epochs...


Epoch 1/2:   0%|          | 0/45 [00:00<?, ?it/s]

ValueError: too many values to unpack (expected 3)

In [ ]:

# Evaluate the model
model.eval()  # Set the model to evaluation mode
test_loss = 0.0
with torch.no_grad(): # Disable gradient calculation during evaluation
    for images, labels in tqdm(test_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels = images.to(device), labels.to(device).unsqueeze(1) # Move data to device and add a dimension to labels

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item() * images.size(0)

test_loss /= len(test_dataset)

print(f"Test Loss (MSE): {test_loss:.4f}")

Epoch 5/5:   0%|          | 0/10 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f314a908720>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1618, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1601, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.11/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f314a908720>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 1618, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py", line 16

Test Loss (MSE): 0.0000


In [ ]:
## Inference

img_path = "/content/AI/192/192_Honor X9C_F3.jpg"

# Ensure the model is on the correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Set the model to evaluation mode
model.eval()

# Load and transform the image
image = Image.open(img_path).convert('RGB')
# Use the same transform as used for the training data
# Assuming 'transform' is still defined from the dataset creation cell
image = transform(image).unsqueeze(0).to(device) # Add batch dimension and move to device

# Perform inference
with torch.no_grad():
    predicted_scaled_value = model(image)

# Inverse scale the prediction to get the original specific gravity value
predicted_specific_gravity = (predicted_scaled_value + 1000) / 1000

print(f"The predicted Sp.Refractometer value for the image {img_path} is: {predicted_specific_gravity.item():.4f}")

The predicted Sp.Refractometer value for the image /content/AI/192/192_Honor X9C_F3.jpg is: 1.0157


# Task
Modify the code to include brightness as a feature for training the CNN model.

## Modify the model

### Subtask:
Update the `CNNRegressor` class to accept the brightness feature as an additional input and adjust the linear regression layer accordingly.


**Reasoning**:
The subtask requires modifying the `CNNRegressor` class to incorporate the brightness feature alongside the image and RGB values. This involves updating the linear regression layer to accept the concatenated features and modifying the forward pass to include the brightness in the concatenation. I will modify the existing `CNNRegressor` class definition to achieve this.



In [ ]:
import torch
import torch.nn as nn
import timm

# Define the CNN model using ConvNextV2 from timm
class CNNRegressor(nn.Module):
    def __init__(self, model_name='timm/convnextv2_base.fcmae_ft_in22k_in1k', pretrained=True):
        super(CNNRegressor, self).__init__()
        # Load model without the final classification layer
        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        # Add a linear layer for regression that also takes RGB values and brightness
        # The output of the CNN is self.model.num_features, and we have 3 RGB values + 1 brightness feature = 4 additional features
        self.regressor = nn.Linear(self.model.num_features + 4, 1)

    def forward(self, image, rgb_values, brightness_feature):
        x = self.model(image)
        # Concatenate CNN output with RGB values and brightness feature
        x = torch.cat((x, rgb_values, brightness_feature.unsqueeze(1)), dim=1) # Add a dimension to brightness_feature
        x = self.regressor(x)
        return x

# Instantiate the model
model = CNNRegressor()

# Define loss function and optimizer
criterion = nn.MSELoss() # Mean Squared Error for regression
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # Adam optimizer

print("CNN Regressor model created with brightness feature input.")

CNN Regressor model created with brightness feature input.


**Reasoning**:
The `CNNRegressor` class has been updated to accept the brightness feature. The next step is to modify the training loop to pass the brightness feature to the model during training and evaluation.



In [ ]:
# Check if GPU is available and move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

num_epochs = 2

print(f"Training the model on {device} for {num_epochs} epochs...")

from tqdm.auto import tqdm

for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    # Wrap train_dataloader with tqdm for progress tracking
    for images, labels, rgb_values, brightness_feature in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels, rgb_values, brightness_feature = images.to(device), labels.to(device).unsqueeze(1), rgb_values.to(device), brightness_feature.to(device) # Move data to device and add a dimension to labels

        # Scale the labels
        scaled_labels = (labels * 1000) - 1000

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images, rgb_values, brightness_feature)
        loss = criterion(outputs, scaled_labels)  # Use scaled labels for loss calculation

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

print("Training finished.")

# Evaluate the model
model.eval()  # Set the model to evaluation mode
test_loss = 0.0
with torch.no_grad(): # Disable gradient calculation during evaluation
    for images, labels, rgb_values, brightness_feature in test_dataloader:
        images, labels, rgb_values, brightness_feature = images.to(device), labels.to(device).unsqueeze(1), rgb_values.to(device), brightness_feature.to(device) # Move data to device and add a dimension to labels

        # Scale the labels for evaluation
        scaled_labels = (labels * 1000) - 1000

        outputs = model(images, rgb_values, brightness_feature)
        loss = criterion(outputs, scaled_labels) # Use scaled labels for loss calculation
        test_loss += loss.item() * images.size(0)

test_loss /= len(test_dataset)

print(f"Test Loss (MSE): {test_loss:.4f}")

Training the model on cuda for 2 epochs...


Epoch 1/2:   0%|          | 0/45 [00:00<?, ?it/s]

ValueError: not enough values to unpack (expected 4, got 3)

**Reasoning**:
The error message "ValueError: not enough values to unpack (expected 4, got 3)" indicates that the `train_dataloader` is only yielding 3 values per iteration, but the code is trying to unpack 4 values. This is because the `UrineDataset` was modified to return 4 values (image, label, rgb_values, brightness_feature), but the dataloaders were created before this modification. I need to recreate the dataloaders after modifying the `UrineDataset` class.



In [ ]:
# Create separate datasets for training and testing with different transforms
img_directory = 'AI'
train_dataset_full = UrineDataset(df, img_directory, transform=train_transform)
test_dataset_full = UrineDataset(df, img_directory, transform=test_transform)

# Define the split ratio (e.g., 80% for training, 20% for testing)
train_ratio = 0.9
test_ratio = 1 - train_ratio
train_size = int(train_ratio * len(train_dataset_full)) # Use the dataset with training transforms
test_size = len(test_dataset_full) - train_size # Use the dataset with testing transforms

# Split the dataset
train_dataset, test_dataset = random_split(train_dataset_full, [train_size, test_size]) # Split the training dataset with augmentation
# Note: We are splitting the full dataset with training transforms for the training set, and the full dataset with testing transforms for the test set.

# Create DataLoaders for training and testing sets
# Increase num_workers for potentially faster data loading
num_workers = 2 # You can adjust this number based on your system's capabilities
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=num_workers)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=num_workers)

print("DataLoaders recreated with updated dataset.")

DataLoaders recreated with updated dataset.


**Reasoning**:
The dataloaders have been recreated to include the brightness feature. Now, I can re-run the training and evaluation code, which was previously stopped due to the `ValueError`, as the dataloader will now provide the expected 4 values.



In [ ]:
# Check if GPU is available and move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

num_epochs = 2

print(f"Training the model on {device} for {num_epochs} epochs...")

from tqdm.auto import tqdm

for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    # Wrap train_dataloader with tqdm for progress tracking
    for images, labels, rgb_values, brightness_feature in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels, rgb_values, brightness_feature = images.to(device), labels.to(device).unsqueeze(1), rgb_values.to(device), brightness_feature.to(device) # Move data to device and add a dimension to labels

        # Scale the labels
        scaled_labels = (labels * 1000) - 1000

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images, rgb_values, brightness_feature)
        loss = criterion(outputs, scaled_labels)  # Use scaled labels for loss calculation

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

print("Training finished.")

# Evaluate the model
model.eval()  # Set the model to evaluation mode
test_loss = 0.0
with torch.no_grad(): # Disable gradient calculation during evaluation
    for images, labels, rgb_values, brightness_feature in test_dataloader:
        images, labels, rgb_values, brightness_feature = images.to(device), labels.to(device).unsqueeze(1), rgb_values.to(device), brightness_feature.to(device) # Move data to device and add a dimension to labels

        # Scale the labels for evaluation
        scaled_labels = (labels * 1000) - 1000

        outputs = model(images, rgb_values, brightness_feature)
        loss = criterion(outputs, scaled_labels) # Use scaled labels for loss calculation
        test_loss += loss.item() * images.size(0)

test_loss /= len(test_dataset)

print(f"Test Loss (MSE): {test_loss:.4f}")

Training the model on cuda for 2 epochs...


Epoch 1/2:   0%|          | 0/45 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 98.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 82.12 MiB is free. Process 5750 has 14.66 GiB memory in use. Of the allocated memory 14.49 GiB is allocated by PyTorch, and 47.41 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Update the training loop retry 1

### Subtask:
Modify the training loop to retrieve the brightness feature from the dataloader and pass it to the model during training.


**Reasoning**:
Modify the training loop to unpack the brightness feature from the dataloader, move it to the device, pass it to the model, and reduce the batch size to avoid the OutOfMemoryError.



In [ ]:
# Check if GPU is available and move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

num_epochs = 2

print(f"Training the model on {device} for {num_epochs} epochs...")

from tqdm.auto import tqdm

# Reduce the batch size to avoid OutOfMemoryError
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=num_workers)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=num_workers)


for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    # Wrap train_dataloader with tqdm for progress tracking
    for images, labels, rgb_values, brightness_feature in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels, rgb_values, brightness_feature = images.to(device), labels.to(device).unsqueeze(1), rgb_values.to(device), brightness_feature.to(device).unsqueeze(1) # Move data to device and add a dimension to labels and brightness feature

        # Scale the labels
        scaled_labels = (labels * 1000) - 1000

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images, rgb_values, brightness_feature)
        loss = criterion(outputs, scaled_labels)  # Use scaled labels for loss calculation

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

print("Training finished.")

# Evaluate the model
model.eval()  # Set the model to evaluation mode
test_loss = 0.0
with torch.no_grad(): # Disable gradient calculation during evaluation
    for images, labels, rgb_values, brightness_feature in test_dataloader:
        images, labels, rgb_values, brightness_feature = images.to(device), labels.to(device).unsqueeze(1), rgb_values.to(device), brightness_feature.to(device).unsqueeze(1) # Move data to device and add a dimension to labels and brightness feature

        # Scale the labels for evaluation
        scaled_labels = (labels * 1000) - 1000

        outputs = model(images, rgb_values, brightness_feature)
        loss = criterion(outputs, scaled_labels) # Use scaled labels for loss calculation
        test_loss += loss.item() * images.size(0)

test_loss /= len(test_dataset)

print(f"Test Loss (MSE): {test_loss:.4f}")

Training the model on cuda for 2 epochs...


Epoch 1/2:   0%|          | 0/90 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 50.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 32.12 MiB is free. Process 5750 has 14.71 GiB memory in use. Of the allocated memory 14.55 GiB is allocated by PyTorch, and 29.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

**Reasoning**:
The OutOfMemoryError persists even after reducing the batch size. This indicates that the model size might still be too large for the available GPU memory. To address this, further reduce the batch size and potentially try a smaller ConvNeXtV2 model variant if available.



In [ ]:
# Check if GPU is available and move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

num_epochs = 2

print(f"Training the model on {device} for {num_epochs} epochs...")

from tqdm.auto import tqdm

# Further reduce the batch size to potentially avoid OutOfMemoryError
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=num_workers)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=num_workers)


for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    # Wrap train_dataloader with tqdm for progress tracking
    for images, labels, rgb_values, brightness_feature in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels, rgb_values, brightness_feature = images.to(device), labels.to(device).unsqueeze(1), rgb_values.to(device), brightness_feature.to(device).unsqueeze(1) # Move data to device and add a dimension to labels and brightness feature

        # Scale the labels
        scaled_labels = (labels * 1000) - 1000

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images, rgb_values, brightness_feature)
        loss = criterion(outputs, scaled_labels)  # Use scaled labels for loss calculation

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

print("Training finished.")

# Evaluate the model
model.eval()  # Set the model to evaluation mode
test_loss = 0.0
with torch.no_grad(): # Disable gradient calculation during evaluation
    for images, labels, rgb_values, brightness_feature in test_dataloader:
        images, labels, rgb_values, brightness_feature = images.to(device), labels.to(device).unsqueeze(1), rgb_values.to(device), brightness_feature.to(device).unsqueeze(1) # Move data to device and add a dimension to labels and brightness feature

        # Scale the labels for evaluation
        scaled_labels = (labels * 1000) - 1000

        outputs = model(images, rgb_values, brightness_feature)
        loss = criterion(outputs, scaled_labels) # Use scaled labels for loss calculation
        test_loss += loss.item() * images.size(0)

test_loss /= len(test_dataset)

print(f"Test Loss (MSE): {test_loss:.4f}")

Training the model on cuda for 2 epochs...


Epoch 1/2:   0%|          | 0/180 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 26.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 6.12 MiB is free. Process 5750 has 14.73 GiB memory in use. Of the allocated memory 14.59 GiB is allocated by PyTorch, and 21.34 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

**Reasoning**:
The OutOfMemoryError persists even with a batch size of 16. This indicates that the model itself is too large for the available GPU memory with the current setup. To potentially resolve this, I will switch to a smaller pre-trained model from the timm library. I will choose 'timm/convnextv2_tiny.fcmae_ft_in22k_in1k' which is a smaller variant of ConvNeXtV2. I will also keep the batch size at 16.



In [ ]:
# Define the CNN model using a smaller ConvNextV2 variant from timm
class CNNRegressor(nn.Module):
    def __init__(self, model_name='timm/convnextv2_tiny.fcmae_ft_in22k_in1k', pretrained=True):
        super(CNNRegressor, self).__init__()
        # Load model without the final classification layer
        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        # Add a linear layer for regression that also takes RGB values and brightness
        # The output of the CNN is self.model.num_features, and we have 3 RGB values + 1 brightness feature = 4 additional features
        self.regressor = nn.Linear(self.model.num_features + 4, 1)

    def forward(self, image, rgb_values, brightness_feature):
        x = self.model(image)
        # Concatenate CNN output with RGB values and brightness feature
        x = torch.cat((x, rgb_values, brightness_feature.unsqueeze(1)), dim=1) # Add a dimension to brightness_feature
        x = self.regressor(x)
        return x

# Instantiate the model with the smaller variant
model = CNNRegressor()

# Define loss function and optimizer
criterion = nn.MSELoss() # Mean Squared Error for regression
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # Adam optimizer

print("CNN Regressor model created with a smaller ConvNeXtV2 variant.")

# Check if GPU is available and move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

num_epochs = 2

print(f"Training the model on {device} for {num_epochs} epochs...")

from tqdm.auto import tqdm

# Keep the reduced batch size
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=num_workers)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=num_workers)


for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    # Wrap train_dataloader with tqdm for progress tracking
    for images, labels, rgb_values, brightness_feature in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels, rgb_values, brightness_feature = images.to(device), labels.to(device).unsqueeze(1), rgb_values.to(device), brightness_feature.to(device).unsqueeze(1) # Move data to device and add a dimension to labels and brightness feature

        # Scale the labels
        scaled_labels = (labels * 1000) - 1000

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images, rgb_values, brightness_feature)
        loss = criterion(outputs, scaled_labels)  # Use scaled labels for loss calculation

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

print("Training finished.")

# Evaluate the model
model.eval()  # Set the model to evaluation mode
test_loss = 0.0
with torch.no_grad(): # Disable gradient calculation during evaluation
    for images, labels, rgb_values, brightness_feature in test_dataloader:
        images, labels, rgb_values, brightness_feature = images.to(device), labels.to(device).unsqueeze(1), rgb_values.to(device), brightness_feature.to(device).unsqueeze(1) # Move data to device and add a dimension to labels and brightness feature

        # Scale the labels for evaluation
        scaled_labels = (labels * 1000) - 1000

        outputs = model(images, rgb_values, brightness_feature)
        loss = criterion(outputs, scaled_labels) # Use scaled labels for loss calculation
        test_loss += loss.item() * images.size(0)

test_loss /= len(test_dataset)

print(f"Test Loss (MSE): {test_loss:.4f}")

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

CNN Regressor model created with a smaller ConvNeXtV2 variant.
Training the model on cuda for 2 epochs...


Epoch 1/2:   0%|          | 0/180 [00:00<?, ?it/s]

RuntimeError: Tensors must have same number of dimensions: got 2 and 3

**Reasoning**:
The RuntimeError "Tensors must have same number of dimensions: got 2 and 3" during concatenation suggests that the dimensions of `x` (the output of the CNN model), `rgb_values`, and `brightness_feature` are not compatible for concatenation along `dim=1`. The CNN output `x` likely has a shape of `(batch_size, features)`, which is 2 dimensions. `rgb_values` also has a shape of `(batch_size, 3)`, which is 2 dimensions. However, `brightness_feature` is currently being unsqueezed to have 3 dimensions `(batch_size, 1, 1)` before being concatenated, which is incorrect. It should be reshaped to `(batch_size, 1)`. I need to fix the concatenation step in the `forward` method of the `CNNRegressor` to ensure all tensors have two dimensions before concatenating along `dim=1`. I also need to remove the extra unsqueeze in the training and evaluation loops for `brightness_feature`.



In [ ]:
# Define the CNN model using a smaller ConvNextV2 variant from timm
class CNNRegressor(nn.Module):
    def __init__(self, model_name='timm/convnextv2_tiny.fcmae_ft_in22k_in1k', pretrained=True):
        super(CNNRegressor, self).__init__()
        # Load model without the final classification layer
        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        # Add a linear layer for regression that also takes RGB values and brightness
        # The output of the CNN is self.model.num_features, and we have 3 RGB values + 1 brightness feature = 4 additional features
        self.regressor = nn.Linear(self.model.num_features + 4, 1)

    def forward(self, image, rgb_values, brightness_feature):
        x = self.model(image)
        # Concatenate CNN output with RGB values and brightness feature
        # Ensure all tensors have 2 dimensions before concatenating along dim=1
        x = torch.cat((x, rgb_values, brightness_feature.unsqueeze(1)), dim=1) # Add a dimension to brightness_feature if it's a scalar
        x = self.regressor(x)
        return x

# Instantiate the model with the smaller variant
model = CNNRegressor()

# Define loss function and optimizer
criterion = nn.MSELoss() # Mean Squared Error for regression
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # Adam optimizer

print("CNN Regressor model created with a smaller ConvNeXtV2 variant and fixed concatenation.")

# Check if GPU is available and move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

num_epochs = 2

print(f"Training the model on {device} for {num_epochs} epochs...")

from tqdm.auto import tqdm

# Keep the reduced batch size
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=num_workers)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=num_workers)


for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    # Wrap train_dataloader with tqdm for progress tracking
    for images, labels, rgb_values, brightness_feature in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        # Remove the extra unsqueeze for brightness_feature here
        images, labels, rgb_values, brightness_feature = images.to(device), labels.to(device).unsqueeze(1), rgb_values.to(device), brightness_feature.to(device) # Move data to device and add a dimension to labels

        # Scale the labels
        scaled_labels = (labels * 1000) - 1000

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images, rgb_values, brightness_feature)
        loss = criterion(outputs, scaled_labels)  # Use scaled labels for loss calculation

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

print("Training finished.")

# Evaluate the model
model.eval()  # Set the model to evaluation mode
test_loss = 0.0
with torch.no_grad(): # Disable gradient calculation during evaluation
    for images, labels, rgb_values, brightness_feature in test_dataloader:
        # Remove the extra unsqueeze for brightness_feature here
        images, labels, rgb_values, brightness_feature = images.to(device), labels.to(device).unsqueeze(1), rgb_values.to(device), brightness_feature.to(device) # Move data to device and add a dimension to labels

        # Scale the labels for evaluation
        scaled_labels = (labels * 1000) - 1000

        outputs = model(images, rgb_values, brightness_feature)
        loss = criterion(outputs, scaled_labels) # Use scaled labels for loss calculation
        test_loss += loss.item() * images.size(0)

test_loss /= len(test_dataset)

print(f"Test Loss (MSE): {test_loss:.4f}")

CNN Regressor model created with a smaller ConvNeXtV2 variant and fixed concatenation.
Training the model on cuda for 2 epochs...


Epoch 1/2:   0%|          | 0/180 [00:00<?, ?it/s]

RuntimeError: CUDA error: CUBLAS_STATUS_ALLOC_FAILED when calling `cublasCreate(handle)`

**Reasoning**:
The CUDA error `CUBLAS_STATUS_ALLOC_FAILED` during the backward pass indicates a failure in allocating memory for matrix multiplications on the GPU. Even with a smaller model and reduced batch size, the memory requirement during backpropagation seems to exceed the available GPU memory. To further reduce memory usage, I will try reducing the image size used for training and evaluation. This will reduce the memory footprint of the images and the intermediate activations in the CNN. I will reduce the image size from 224x224 to 128x128.



In [ ]:
# Define transformations for training with augmentation and smaller image size
train_transform = transforms.Compose([
    transforms.Resize((128, 128)), # Reduce image size
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Define transformations for testing (no augmentation) with smaller image size
test_transform = transforms.Compose([
    transforms.Resize((128, 128)), # Reduce image size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create separate datasets for training and testing with the updated transforms
img_directory = 'AI'
train_dataset_full = UrineDataset(df, img_directory, transform=train_transform)
test_dataset_full = UrineDataset(df, img_directory, transform=test_transform)

# Define the split ratio (e.g., 80% for training, 20% for testing)
train_ratio = 0.9
test_ratio = 1 - train_ratio
train_size = int(train_ratio * len(train_dataset_full)) # Use the dataset with training transforms
test_size = len(test_dataset_full) - train_size # Use the dataset with testing transforms

# Split the dataset
train_dataset, test_dataset = random_split(train_dataset_full, [train_size, test_size]) # Split the training dataset with augmentation
# Note: We are splitting the full dataset with training transforms for the training set, and the full dataset with testing transforms for the test set.

# Create DataLoaders for training and testing sets
# Keep the reduced batch size
num_workers = 2
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=num_workers)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=num_workers)


print("DataLoaders recreated with smaller image size.")

# Check if GPU is available and move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

num_epochs = 2

print(f"Training the model on {device} for {num_epochs} epochs...")

from tqdm.auto import tqdm

for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    # Wrap train_dataloader with tqdm for progress tracking
    for images, labels, rgb_values, brightness_feature in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels, rgb_values, brightness_feature = images.to(device), labels.to(device).unsqueeze(1), rgb_values.to(device), brightness_feature.to(device).unsqueeze(1) # Move data to device and add a dimension to labels and brightness feature

        # Scale the labels
        scaled_labels = (labels * 1000) - 1000

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images, rgb_values, brightness_feature)
        loss = criterion(outputs, scaled_labels)  # Use scaled labels for loss calculation

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

print("Training finished.")

# Evaluate the model
model.eval()  # Set the model to evaluation mode
test_loss = 0.0
with torch.no_grad(): # Disable gradient calculation during evaluation
    for images, labels, rgb_values, brightness_feature in test_dataloader:
        images, labels, rgb_values, brightness_feature = images.to(device), labels.to(device).unsqueeze(1), rgb_values.to(device), brightness_feature.to(device).unsqueeze(1) # Move data to device and add a dimension to labels and brightness feature

        # Scale the labels for evaluation
        scaled_labels = (labels * 1000) - 1000

        outputs = model(images, rgb_values, brightness_feature)
        loss = criterion(outputs, scaled_labels) # Use scaled labels for loss calculation
        test_loss += loss.item() * images.size(0)

test_loss /= len(test_dataset)

print(f"Test Loss (MSE): {test_loss:.4f}")

DataLoaders recreated with smaller image size.
Training the model on cuda for 2 epochs...


Epoch 1/2:   0%|          | 0/180 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7a92fca73600>
Exception ignored in: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7a92fca73600>    
self._shutdown_workers()Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__

      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
self._shutdown_workers()    
if w.is_alive():  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers

      if w.is_alive(): 
          ^ ^^^^^^^^^^^^^^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    
assert self._parent_pid == os.getpid(), 'can only test a child process'  File "/usr/lib/python3

RuntimeError: Tensors must have same number of dimensions: got 2 and 3

**Reasoning**:
The RuntimeError "Tensors must have same number of dimensions: got 2 and 3" reappeared, indicating the concatenation issue was not fully resolved. Looking at the traceback and the previous code, the `brightness_feature` is still being unsqueezed inside the `forward` method, which is likely the cause of the dimension mismatch when concatenating with the 2D output of the CNN and RGB values. The `brightness_feature` should already be a 1D tensor of shape `(batch_size,)` from the dataloader, and it needs to be reshaped to `(batch_size, 1)` before concatenation. I will remove the `unsqueeze(1)` call inside the `forward` method and ensure the `brightness_feature` is correctly handled as a 1D tensor from the dataloader that is then reshaped to 2D `(batch_size, 1)` for concatenation. I will also remove the extra unsqueeze in the training and evaluation loops as it is handled in the forward pass.



In [ ]:
# Define the CNN model using a smaller ConvNextV2 variant from timm
class CNNRegressor(nn.Module):
    def __init__(self, model_name='timm/convnextv2_tiny.fcmae_ft_in22k_in1k', pretrained=True):
        super(CNNRegressor, self).__init__()
        # Load model without the final classification layer
        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        # Add a linear layer for regression that also takes RGB values and brightness
        # The output of the CNN is self.model.num_features, and we have 3 RGB values + 1 brightness feature = 4 additional features
        self.regressor = nn.Linear(self.model.num_features + 4, 1)

    def forward(self, image, rgb_values, brightness_feature):
        x = self.model(image)
        # Concatenate CNN output with RGB values and brightness feature
        # Ensure brightness_feature is reshaped to (batch_size, 1) for concatenation
        x = torch.cat((x, rgb_values, brightness_feature.view(-1, 1)), dim=1) # Reshape brightness_feature to (batch_size, 1)
        x = self.regressor(x)
        return x

# Instantiate the model with the smaller variant
model = CNNRegressor()

# Define loss function and optimizer
criterion = nn.MSELoss() # Mean Squared Error for regression
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # Adam optimizer

print("CNN Regressor model created with a smaller ConvNeXtV2 variant and corrected concatenation.")

# Check if GPU is available and move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

num_epochs = 2

print(f"Training the model on {device} for {num_epochs} epochs...")

from tqdm.auto import tqdm

# Keep the reduced batch size and smaller image size
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=num_workers)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=num_workers)


for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    # Wrap train_dataloader with tqdm for progress tracking
    for images, labels, rgb_values, brightness_feature in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        # Move data to device and add a dimension to labels. brightness_feature is moved as is.
        images, labels, rgb_values, brightness_feature = images.to(device), labels.to(device).unsqueeze(1), rgb_values.to(device), brightness_feature.to(device)

        # Scale the labels
        scaled_labels = (labels * 1000) - 1000

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images, rgb_values, brightness_feature)
        loss = criterion(outputs, scaled_labels)  # Use scaled labels for loss calculation

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

print("Training finished.")

# Evaluate the model
model.eval()  # Set the model to evaluation mode
test_loss = 0.0
with torch.no_grad(): # Disable gradient calculation during evaluation
    for images, labels, rgb_values, brightness_feature in test_dataloader:
        # Move data to device and add a dimension to labels. brightness_feature is moved as is.
        images, labels, rgb_values, brightness_feature = images.to(device), labels.to(device).unsqueeze(1), rgb_values.to(device), brightness_feature.to(device)

        # Scale the labels for evaluation
        scaled_labels = (labels * 1000) - 1000

        outputs = model(images, rgb_values, brightness_feature)
        loss = criterion(outputs, scaled_labels) # Use scaled labels for loss calculation
        test_loss += loss.item() * images.size(0)

test_loss /= len(test_dataset)

print(f"Test Loss (MSE): {test_loss:.4f}")

CNN Regressor model created with a smaller ConvNeXtV2 variant and corrected concatenation.
Training the model on cuda for 2 epochs...


Epoch 1/2:   0%|          | 0/180 [00:00<?, ?it/s]

Epoch [1/2], Loss: 40.9887


Epoch 2/2:   0%|          | 0/180 [00:00<?, ?it/s]

KeyboardInterrupt: 